# RSS-inator

Prepares a report with LLM-generated summaries of RSS-sourced content.

## Load useful libraries

In [1]:
import markdown
import os
import json
from IPython.display import display, Markdown
from feed_to_markdown_report import go

## Parameters

In [2]:
feeds_list_json_filename = os.environ['HOME'] + '/Desktop/projects/operation-clusterfuck/daily_reports/feeds_to_report.json'
hours_cutoff = 24
model_to_use = 'llama3.2:3b'
display_the_markdown = False
output_html_filename = os.environ['HOME'] + '/Desktop/daily_report.html'
output_markdown_filename = os.environ['HOME'] + '/Desktop/daily_report.md'

## Load list of feeds we want to report

In [3]:
with open(feeds_list_json_filename) as f:
    list_feeds = json.load(f)['feeds']

In [4]:
list_feeds

[{'title': 'The Register: AI & ML',
  'url': 'https://api.theregister.com/api/v1/article?query=tag:%22ai%20and%20ml%22&orderBy=published&site_id=2&remapper=rss'},
 {'title': 'The Register: Offbeat',
  'url': 'https://api.theregister.com/api/v1/article?query=tag:offbeat&orderBy=published&site_id=2&remapper=rss'},
 {'title': 'AI News RSS Feed',
  'url': 'https://www.artificialintelligence-news.com/feed/'},
 {'title': 'AI Trends - The Best Source for AI News and Events RSS Feed',
  'url': 'https://www.aitrends.com/feed/'},
 {'title': 'DailyAI RSS Feed', 'url': 'https://dailyai.com/feed/'},
 {'title': 'The Rundown AI RSS Feed',
  'url': 'https://rss.beehiiv.com/feeds/2R3C6Bt5wj.xml'},
 {'title': 'AI Insider RSS Feed', 'url': 'https://theaiinsider.tech/feed/'},
 {'title': 'AIwire RSS Feed', 'url': 'https://www.aiwire.net/feed'},
 {'title': 'MIT News - Artificial intelligence RSS Feed',
  'url': 'http://news.mit.edu/rss/topic/artificial-intelligence2'},
 {'title': 'Machine Intelligence Resea

## Define a function to process the feeds

In [5]:
def process_feeds(list_feeds, hours_cutoff = 24, model_to_use = model_to_use):
    list_reports = []
    for feed in list_feeds:
        list_reports.append(
            go(
                feed['url'],
                feed['title'],
                hours_cutoff = hours_cutoff,
                model_to_use = model_to_use,
            )
        )
    return list_reports

## Generate report

In [6]:
list_reports = process_feeds(list_feeds, hours_cutoff = hours_cutoff, model_to_use = model_to_use)

In [7]:
print(len(list_reports))

49


In [103]:
pre_final_report = ''.join(list_reports)

## Summarize the full report

In [120]:
import ollama

def generate_strategy_summary(report, model_to_use = 'llama3.2:3b'):

    prompt = """You are an expert strategic analyst specializing in AI/ML strategy,
    business development, and career strategy for senior technical professionals.

You will be given an arbitrary Markdown document (e.g., notes, reports, article 
summaries, research digests, project logs, or strategic memos).
Your task is to produce a **holistic strategic synthesis** of the document, emphasizing:
- business strategy implications (market trends, competitive positioning, risks, opportunities, commercialization angles)
- career strategy implications (skills to build, roles to target, portfolio positioning, signaling, networking, research directions)

Rules:
- Output MUST be valid Markdown.
- Output MUST contain ONLY 3–5 bullet points.
- Each bullet point must be 1–2 sentences maximum.
- Do NOT include headings, preambles, conclusions, or any other text.
- Do NOT quote the document directly; abstract and generalize.
- Prioritize the highest-leverage insights (what matters most, not what appears most often).
- Explicitly call out strategic opportunities, threats, and actionable positioning moves.

Now analyze and summarize the following Markdown document:
        
    """
    
    prompt += report

    response: ollama.ChatResponse = ollama.chat(
        model = model_to_use,
        messages = [
          {
            'role': 'user',
            'content': prompt,
          },
        ]
    )
    return response.message.content

In [125]:
strategy_analysis = generate_strategy_summary(pre_final_report, model_to_use = model_to_use)

In [130]:
final_report = '# Overall Strategic Summary\n' + strategy_analysis + '\n\n# Individual RSS Feed Summaries\n\n' + pre_final_report.replace('# ', '## ')

## Save final report as HTML

In [131]:
html = markdown.markdown(final_report)
with open(output_html_filename, 'w') as f:
    f.write(html.replace('\\$', '$') + '\n')

## Save final report as Markdown

In [109]:
with open(output_markdown_filename, 'w') as f:
    f.write(final_report + '\n')

## Optionally display the Markdown here

In [11]:
if display_the_markdown:
    display(Markdown(final_report))